Tensorflow

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

Mnist -> fashion

In [2]:
#mnist цифры

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = x_train.reshape(x_train.shape[0], -1)
X_test = x_test.reshape(x_test.shape[0], -1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
X_train shape: (60000, 784)
X_test shape: (10000, 784)


In [3]:
def build_model(input_shape, neurons1=128, neurons2=64, dropout_rate=0.2):
    model = Sequential([
        Dense(neurons1, activation='relu', input_shape=(input_shape,)),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(neurons2, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [4]:
model = build_model(X_train.shape[1])

start_time = time.time()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    verbose=1
)

end_time = time.time()
total_time = end_time - start_time

y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

report = classification_report(y_test, y_pred)
print(f"Metrics: {report}")

print(f"Время обучения: {total_time/60:.2f} минут")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.8421 - loss: 0.5369 - val_accuracy: 0.9388 - val_loss: 0.2125
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9298 - loss: 0.2374 - val_accuracy: 0.9517 - val_loss: 0.1620
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9451 - loss: 0.1824 - val_accuracy: 0.9606 - val_loss: 0.1348
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9545 - loss: 0.1466 - val_accuracy: 0.9656 - val_loss: 0.1202
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9609 - loss: 0.1257 - val_accuracy: 0.9662 - val_loss: 0.1138
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9648 - loss: 0.1128 - val_accuracy: 0.9689 - val_loss: 0.1091
Epoch 7/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9683 - loss: 0.1006 - val_accuracy: 0.9702 - val_loss: 0.1061
Epoch 8/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9726 - loss: 0.0883 - val_accuracy: 0

In [5]:
#mnist fashion

(x_fashion_train, y_fashion_train), (x_fashion_test, y_fashion_test) = tf.keras.datasets.fashion_mnist.load_data()

selected_classes = [0, 1, 2, 3, 4]

train_mask = np.isin(y_fashion_train, selected_classes)
test_mask = np.isin(y_fashion_test, selected_classes)

X_fashion_train = x_fashion_train[train_mask]
y_fashion_train = y_fashion_train[train_mask]
X_fashion_test = x_fashion_test[test_mask]
y_fashion_test = y_fashion_test[test_mask]

X_fashion_train = X_fashion_train.reshape(X_fashion_train.shape[0], -1)
X_fashion_test = X_fashion_test.reshape(X_fashion_test.shape[0], -1)

X_fashion_train = scaler.transform(X_fashion_train)
X_fashion_test = scaler.transform(X_fashion_test)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Без разморозки тела

In [6]:
fashion_model = Sequential()

for layer in model.layers[:-1]:
    fashion_model.add(layer)

for layer in fashion_model.layers:
    layer.trainable = False

fashion_model.add(
    Dense(32, activation='relu', kernel_initializer='he_normal')
)

fashion_model.add(BatchNormalization())

fashion_model.add(Dropout(0.2))

fashion_model.add(
    Dense(5, activation='softmax')
)

fashion_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Обучение только новой головы на Fashion MNIST")

history_transfer = fashion_model.fit(
    X_fashion_train, y_fashion_train,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    verbose=1
)

y_pred_proba_fashion = fashion_model.predict(X_fashion_test, verbose=0)
y_pred_fashion = np.argmax(y_pred_proba_fashion, axis=1)

accuracy_transfer = np.mean(y_pred_fashion == y_fashion_test)

print(f"\nРезультаты Transfer Learning:")
print(f"Точность на тесте: {accuracy_transfer:.4f}")
print(
    classification_report(
        y_fashion_test,
        y_pred_fashion
    )
)

Обучение только новой головы на Fashion MNIST
Epoch 1/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - accuracy: 0.2775 - loss: 1.8537 - val_accuracy: 0.3890 - val_loss: 1.4014
Epoch 2/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3519 - loss: 1.5319 - val_accuracy: 0.4470 - val_loss: 1.3510
Epoch 3/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3823 - loss: 1.4487 - val_accuracy: 0.4447 - val_loss: 1.3389
Epoch 4/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3912 - loss: 1.4180 - val_accuracy: 0.4533 - val_loss: 1.3283
Epoch 5/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4003 - loss: 1.3922 - val_accuracy: 0.4532 - val_loss: 1.3246
Epoch 6/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4163 - loss: 1.3712 - val_accuracy: 0.4443 - val_loss: 1.3223
Epoch 7/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4192 - loss: 1.3708 - val_accuracy: 0.4525 - val_loss: 1.3166
Epoch 8/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4238 

С разморозкой тела

In [7]:
for layer in fashion_model.layers:
    layer.trainable = True

fashion_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = fashion_model.fit(
    X_fashion_train,
    y_fashion_train,

    validation_split=0.2,

    epochs=20,
    batch_size=256,

    verbose=1
)

y_pred_proba_fashion = fashion_model.predict(
    X_fashion_test,
    verbose=0
)

y_pred_fashion = np.argmax(
    y_pred_proba_fashion,
    axis=1
)

accuracy_finetune = np.mean(
    y_pred_fashion == y_fashion_test
)

print(
    f"Accuracy после разморозки тела: "
    f"{accuracy_finetune:.4f}"
)


print("\nClassification Report:")

print(
    classification_report(
        y_fashion_test,
        y_pred_fashion
    )
)

Epoch 1/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.3919 - loss: 1.3962 - val_accuracy: 0.3240 - val_loss: 1.4675
Epoch 2/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5200 - loss: 1.1493 - val_accuracy: 0.4858 - val_loss: 1.2813
Epoch 3/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6047 - loss: 1.0026 - val_accuracy: 0.6558 - val_loss: 1.1068
Epoch 4/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6583 - loss: 0.8987 - val_accuracy: 0.7205 - val_loss: 0.9608
Epoch 5/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6942 - loss: 0.8257 - val_accuracy: 0.7410 - val_loss: 0.8263
Epoch 6/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7183 - loss: 0.7617 - val_accuracy: 0.7623 - val_loss: 0.7277
Epoch 7/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7469 - loss: 0.7058 - val_accuracy: 0.7890 - val_loss: 0.6500
Epoch 8/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7682 - loss: 0.6614 - val_accuracy: 0.8058 - val_los

Fashion -> mnist

In [8]:
#mnist fashion

(x_fashion_train, y_fashion_train), (x_fashion_test, y_fashion_test) = tf.keras.datasets.fashion_mnist.load_data()

selected_classes = [0, 1, 2, 3, 4]

train_mask = np.isin(y_fashion_train, selected_classes)
test_mask = np.isin(y_fashion_test, selected_classes)

X_fashion_train = x_fashion_train[train_mask]
y_fashion_train = y_fashion_train[train_mask]
X_fashion_test = x_fashion_test[test_mask]
y_fashion_test = y_fashion_test[test_mask]

X_fashion_train = X_fashion_train.reshape(X_fashion_train.shape[0], -1)
X_fashion_test = X_fashion_test.reshape(X_fashion_test.shape[0], -1)

scaler = StandardScaler()
X_fashion_train = scaler.fit_transform(X_fashion_train)
X_fashion_test = scaler.transform(X_fashion_test)

In [9]:
def build_model(input_shape, neurons1=128, neurons2=64, dropout_rate=0.2):
    model = Sequential([
        Dense(neurons1, activation='relu', input_shape=(input_shape,)),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(neurons2, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [10]:
model = build_model(X_fashion_train.shape[1])

start_time = time.time()

history = model.fit(
    X_fashion_train, y_fashion_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    verbose=1
)

end_time = time.time()
total_time = end_time - start_time

y_pred_proba = model.predict(X_fashion_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

report = classification_report(y_fashion_test, y_pred)
print(f"Metrics: {report}")

print(f"Время обучения: {total_time/60:.2f} минут")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.8234 - loss: 0.4897 - val_accuracy: 0.8805 - val_loss: 0.3393
Epoch 2/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8807 - loss: 0.3326 - val_accuracy: 0.8960 - val_loss: 0.2882
Epoch 3/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8958 - loss: 0.2897 - val_accuracy: 0.8983 - val_loss: 0.2738
Epoch 4/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9027 - loss: 0.2681 - val_accuracy: 0.9057 - val_loss: 0.2630
Epoch 5/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9075 - loss: 0.2529 - val_accuracy: 0.9027 - val_loss: 0.2593
Epoch 6/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9095 - loss: 0.2419 - val_accuracy: 0.9093 - val_loss: 0.2514
Epoch 7/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9143 - loss: 0.2301 - val_accuracy: 0.9070 - val_loss: 0.2495
Epoch 8/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9214 - loss: 0.2149 - val_accuracy: 0.9115 - val_loss

In [11]:
#mnist цифры

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = x_train.reshape(x_train.shape[0], -1)
X_test = x_test.reshape(x_test.shape[0], -1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (60000, 784)
X_test shape: (10000, 784)


Без разморозки тела

In [12]:
mnist_model = Sequential()

for layer in fashion_model.layers[:-1]:
    mnist_model.add(layer)

for layer in mnist_model.layers:
    layer.trainable = False

mnist_model.add(
    Dense(32, activation='relu', kernel_initializer='he_normal')
)

mnist_model.add(BatchNormalization())

mnist_model.add(Dropout(0.2))

mnist_model.add(
    Dense(10, activation='softmax')
)

mnist_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Обучение только новой головы на MNIST")

history_transfer = mnist_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    verbose=1
)

y_pred_proba = mnist_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

accuracy_transfer = np.mean(y_pred == y_test)

print(f"\nРезультаты Transfer Learning:")
print(f"Точность на тесте: {accuracy_transfer:.4f}")
print(
    classification_report(
        y_test,
        y_pred
    )
)

Обучение только новой головы на MNIST
Epoch 1/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.1872 - loss: 2.4021 - val_accuracy: 0.3848 - val_loss: 1.8620
Epoch 2/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2574 - loss: 2.0807 - val_accuracy: 0.4453 - val_loss: 1.7195
Epoch 3/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2789 - loss: 2.0193 - val_accuracy: 0.4917 - val_loss: 1.6674
Epoch 4/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2877 - loss: 1.9928 - val_accuracy: 0.5025 - val_loss: 1.6419
Epoch 5/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2974 - loss: 1.9729 - val_accuracy: 0.5061 - val_loss: 1.6256
Epoch 6/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3020 - loss: 1.9622 - val_accuracy: 0.5236 - val_loss: 1.6112
Epoch 7/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3052 - loss: 1.9544 - val_accuracy: 0.5166 - val_loss: 1.6129
Epoch 8/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0

С разморозкой тела

In [13]:
for layer in mnist_model.layers:
    layer.trainable = True

mnist_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = mnist_model.fit(
    X_train,
    y_train,

    validation_split=0.2,

    epochs=20,
    batch_size=256,

    verbose=1
)

y_pred_proba = mnist_model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_pred_proba,
    axis=1
)

accuracy_finetune = np.mean(
    y_pred == y_test
)

print(
    f"Accuracy после разморозки тела: "
    f"{accuracy_finetune:.4f}"
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.6306 - loss: 1.3419 - val_accuracy: 0.2540 - val_loss: 1.9529
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7685 - loss: 1.0307 - val_accuracy: 0.5122 - val_loss: 1.5117
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8542 - loss: 0.7962 - val_accuracy: 0.8697 - val_loss: 0.7900
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9001 - loss: 0.6212 - val_accuracy: 0.9427 - val_loss: 0.4361
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9234 - loss: 0.4944 - val_accuracy: 0.9574 - val_loss: 0.2985
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9385 - loss: 0.4013 - val_accuracy: 0.9625 - val_loss: 0.2332
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9461 - loss: 0.3369 - val_accuracy: 0.9647 - val_loss: 0.1927
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9545 - loss: 0.2826 - val_accuracy: 